# LeetCode 2570 — Merge Two 2D Arrays by Summing Values bằng Python

Link đề: https://leetcode.com/problems/merge-two-2d-arrays-by-summing-values/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Two Pointers

Skeleton LeetCode:

```python
class Solution:
    def mergeArrays(self, nums1: List[List[int]], nums2: List[List[int]]) -> List[List[int]]:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: hai danh sách các cặp `[id, val]`, mỗi `id` là duy nhất trong từng danh sách; ta phải **gộp** theo `id` và **cộng giá trị**, kết quả sắp theo `id` tăng dần.
2. Biết đọc/ghi cấu trúc "list chứa các list nhỏ" (`[[1, 2], [2, 3]]`) và "bung" từng cặp bằng `for key, val in nums1`.
3. Dùng **dict** làm bảng cộng dồn (`d[key] = d.get(key, 0) + val`) rồi `sorted(...)` để có kết quả theo thứ tự.
4. Tận dụng việc **cả hai mảng đã sắp sẵn theo id** để gộp bằng **hai con trỏ** như bước "merge" của merge sort, không cần sắp xếp lại.
5. Viết 4 cách (vét cạn, dict, hai con trỏ, mảng đếm theo chỉ số), so sánh và đối chiếu bằng test ngẫu nhiên có seed.
6. Tránh các lỗi quen thuộc: `update` ghi đè thay vì cộng, quên sắp xếp, quên phần dư sau vòng `while`, trả về tuple thay vì list.

## 2. Đề bài & ví dụ

Cho hai mảng 2 chiều `nums1` và `nums2`.

- `nums1[i] = [id_i, val_i]` nghĩa là số có **mã** `id_i` mang **giá trị** `val_i`.
- `nums2[i] = [id_i, val_i]` tương tự.
- Mỗi mảng có các `id` **không trùng nhau** và đã được **sắp tăng dần theo id**.

Hãy gộp hai mảng thành **một** mảng, sắp tăng dần theo `id`, thoả:

- Chỉ chứa những `id` xuất hiện trong **ít nhất một** trong hai mảng.
- Mỗi `id` chỉ xuất hiện **một lần**, giá trị là **tổng** các giá trị của `id` đó ở hai mảng. Nếu `id` không có ở một mảng, coi giá trị ở mảng đó là `0`.

| Ví dụ | nums1 | nums2 | Output | Giải thích |
|---|---|---|---|---|
| 1 | `[[1,2],[2,3],[4,5]]` | `[[1,4],[3,2],[4,1]]` | `[[1,6],[2,3],[3,2],[4,6]]` | id 1: 2+4=6; id 2: chỉ có ở nums1 → 3; id 3: chỉ có ở nums2 → 2; id 4: 5+1=6 |
| 2 | `[[2,4],[3,6],[5,5]]` | `[[1,3],[4,3]]` | `[[1,3],[2,4],[3,6],[4,3],[5,5]]` | hai mảng không có id chung → chỉ trộn lại theo thứ tự |

Constraints: `1 <= nums1.length, nums2.length <= 200`; `nums1[i].length == nums2[j].length == 2`; `1 <= id_i, val_i <= 1000`; hai mảng đều có id không trùng và **tăng ngặt** theo id.

In [1]:
nums1 = [[1, 2], [2, 3], [4, 5]]
nums2 = [[1, 4], [3, 2], [4, 1]]
print("nums1 có", len(nums1), "cặp:", nums1)
print("nums2 có", len(nums2), "cặp:", nums2)
print("kết quả mong đợi theo đề: [[1, 6], [2, 3], [3, 2], [4, 6]]")

nums1 có 3 cặp: [[1, 2], [2, 3], [4, 5]]
nums2 có 3 cặp: [[1, 4], [3, 2], [4, 1]]
kết quả mong đợi theo đề: [[1, 6], [2, 3], [3, 2], [4, 6]]


## 3. Kiến thức nền

### 3.1. List chứa list và việc "bung" từng cặp

`nums1` là một list mà **mỗi phần tử cũng là một list 2 số** `[id, val]`. Python cho phép viết `for key, val in nums1`: mỗi vòng, cặp `[id, val]` được "bung" thành hai biến. Ta tránh đặt tên biến là `id` vì `id` là hàm có sẵn của Python.

In [2]:
nums1 = [[1, 2], [2, 3], [4, 5]]
print(nums1[0], nums1[0][0], nums1[0][1])   # [1, 2] 1 2
for key, val in nums1:
    print("mã", key, "có giá trị", val)

[1, 2] 1 2
mã 1 có giá trị 2
mã 2 có giá trị 3
mã 4 có giá trị 5


### 3.2. `dict` làm bảng cộng dồn

`dict` lưu cặp `key -> value`. Với bài này, `key` là `id`, `value` là tổng đang tích luỹ.

- `d.get(key, 0)` trả `d[key]` nếu có, ngược lại trả `0` (không gây lỗi `KeyError`).
- Muốn **cộng dồn**: `d[key] = d.get(key, 0) + val`.

In [3]:
d = {}
for key, val in [[1, 2], [2, 3], [1, 4]]:
    d[key] = d.get(key, 0) + val
    print("sau khi xử lý", [key, val], "->", d)
# id 1 xuất hiện hai lần nên được cộng: 2 + 4 = 6

sau khi xử lý [1, 2] -> {1: 2}
sau khi xử lý [2, 3] -> {1: 2, 2: 3}
sau khi xử lý [1, 4] -> {1: 6, 2: 3}


### 3.3. `sorted(d.items())` và list comprehension

`d.items()` cho ra các cặp `(key, value)`. `sorted(...)` sắp theo phần tử đầu (tức `key`) tăng dần. List comprehension `[[k, v] for k, v in ...]` đổi mỗi tuple thành list `[k, v]` đúng format đề yêu cầu.

In [4]:
d = {4: 6, 1: 6, 3: 2, 2: 3}
print(list(d.items()))                          # thứ tự chèn: [(4, 6), (1, 6), (3, 2), (2, 3)]
print(sorted(d.items()))                        # tuple, đã sắp theo key
print([[k, v] for k, v in sorted(d.items())])   # list của list, đúng yêu cầu

[(4, 6), (1, 6), (3, 2), (2, 3)]
[(1, 6), (2, 3), (3, 2), (4, 6)]
[[1, 6], [2, 3], [3, 2], [4, 6]]


### 3.4. Hai con trỏ trên hai mảng đã sắp xếp

Khi hai mảng đã sắp tăng, ta giữ hai chỉ số `i` (cho `nums1`) và `j` (cho `nums2`):

- nếu `nums1[i][0] < nums2[j][0]` → id nhỏ hơn chỉ có ở `nums1` (vì `nums2` đã sắp, mọi id còn lại của nó đều >= `nums2[j][0]`) → lấy `nums1[i]`, `i += 1`;
- nếu lớn hơn → ngược lại, lấy `nums2[j]`, `j += 1`;
- nếu bằng nhau → cộng giá trị, cả hai chỉ số cùng tăng.

Đây chính là bước "merge" trong merge sort.

In [5]:
a = [1, 3, 5]
b = [2, 3, 6]
i = j = 0
while i < len(a) and j < len(b):
    if a[i] < b[j]:
        print("lấy", a[i], "từ a");  i += 1
    elif a[i] > b[j]:
        print("lấy", b[j], "từ b");  j += 1
    else:
        print("bằng nhau", a[i], "-> gộp");  i += 1; j += 1
print("phần dư của a:", a[i:], "| phần dư của b:", b[j:])

lấy 1 từ a
lấy 2 từ b
bằng nhau 3 -> gộp
lấy 5 từ a
phần dư của a: [] | phần dư của b: [6]


### 3.5. Mảng đếm theo chỉ số (khi giá trị khoá nhỏ)

Vì `1 <= id <= 1000`, ta có thể dùng một mảng `total` dài `1001` và coi **chỉ số = id**: `total[id] += val`. Cuối cùng duyệt `id` từ `1` đến `1000` thì tự nhiên đã **theo thứ tự tăng dần**, không cần sort.

In [6]:
total = [0] * 11            # minh hoạ với id tối đa 10
for key, val in [[1, 2], [2, 3], [4, 5]] + [[1, 4], [3, 2], [4, 1]]:
    total[key] += val
print(total)
print([[k, total[k]] for k in range(len(total)) if total[k] > 0])

[0, 6, 3, 2, 6, 0, 0, 0, 0, 0, 0]
[[1, 6], [2, 3], [3, 2], [4, 6]]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hãy tưởng tượng hai bảng điểm của hai bài kiểm tra, mỗi học sinh có một mã `id`. Muốn có **bảng tổng điểm**:

1. Liệt kê tất cả mã xuất hiện ở bất kỳ bảng nào.
2. Với từng mã (theo thứ tự tăng dần), tìm điểm ở bảng 1 (không thấy → 0), tìm điểm ở bảng 2 (không thấy → 0), cộng lại.

Tính tay ví dụ 1:

```
id : 1 2 3 4
b1 : 2 3 - 5
b2 : 4 - 2 1
tổng: 6 3 2 6   -> [[1,6],[2,3],[3,2],[4,6]]
```

In [7]:
nums1 = [[1, 2], [2, 3], [4, 5]]
nums2 = [[1, 4], [3, 2], [4, 1]]
for want in [1, 2, 3, 4]:
    v1 = sum(v for k, v in nums1 if k == want)   # không có thì tổng rỗng = 0
    v2 = sum(v for k, v in nums2 if k == want)
    print(f"id={want}: {v1} + {v2} = {v1 + v2}")

id=1: 2 + 4 = 6
id=2: 3 + 0 = 3
id=3: 0 + 2 = 2
id=4: 5 + 1 = 6


## 5. Cách 1 — Brute Force / baseline

Làm đúng như trực giác: gom tập các `id` (dùng `set` để loại trùng), sắp xếp chúng, rồi với **mỗi id** quét tuyến tính cả hai mảng để tìm giá trị tương ứng.

In [8]:
from typing import List

def merge_bruteforce(nums1: List[List[int]], nums2: List[List[int]]) -> List[List[int]]:
    all_ids = set()
    for key, _ in nums1:
        all_ids.add(key)
    for key, _ in nums2:
        all_ids.add(key)

    result = []
    for want in sorted(all_ids):
        s = 0
        for key, val in nums1:          # quét hết nums1
            if key == want:
                s += val
        for key, val in nums2:          # quét hết nums2
            if key == want:
                s += val
        result.append([want, s])
    return result

In [9]:
print(merge_bruteforce([[1, 2], [2, 3], [4, 5]], [[1, 4], [3, 2], [4, 1]]))
print(merge_bruteforce([[2, 4], [3, 6], [5, 5]], [[1, 3], [4, 3]]))

[[1, 6], [2, 3], [3, 2], [4, 6]]
[[1, 3], [2, 4], [3, 6], [4, 3], [5, 5]]


**Độ phức tạp:** gọi `n = len(nums1)`, `m = len(nums2)`, `k` là số id khác nhau (`k <= n + m`). Mỗi id quét `n + m` phần tử → Time `O(k * (n + m))` = `O((n + m)^2)`; Space `O(k)`.

**Nút thắt:** ta quét lại hai mảng **từ đầu** cho mỗi id, dù ta đã biết cách tra id "tức thì" (dict) hoặc biết hai mảng đã sắp (nên chỉ cần đi một lượt). Với `n, m <= 200` cách này vẫn qua, nhưng chưa tốt.

## 6. Ý tưởng thuật toán / bước đột phá

Có hai "phần còn thiếu" để bỏ việc quét đi quét lại:

1. **Tra cứu nhanh bằng dict:** duyệt mỗi mảng **một lần**, cộng dồn `d[key] += val`. Cuối cùng chỉ còn sắp xếp các key. Thời gian `O((n + m) log(n + m))` do bước sort.
2. **Dùng tính chất đã sắp xếp:** vì hai mảng đã tăng theo id, ta "kéo khoá" như khoá kéo: luôn so hai id đầu tiên chưa xử lý, lấy cái nhỏ hơn (hoặc gộp nếu bằng). Mỗi phần tử chỉ đi qua **một lần** → `O(n + m)` mà kết quả tự nhiên đã đúng thứ tự.

Quy tắc "nếu… thì…" của hai con trỏ: `id1 < id2` → lấy bên 1; `id1 > id2` → lấy bên 2; `id1 == id2` → cộng; hết một bên → chép nốt phần còn lại của bên kia.

In [10]:
nums1 = [[1, 2], [2, 3], [4, 5]]
nums2 = [[1, 4], [3, 2], [4, 1]]
# ý tưởng dict: cộng dồn rồi sort
d = {}
for key, val in nums1 + nums2:
    d[key] = d.get(key, 0) + val
print(d)
print(sorted(d.items()))

{1: 6, 2: 3, 4: 6, 3: 2}
[(1, 6), (2, 3), (3, 2), (4, 6)]


## 7. Các cách tiếp theo

Ngoài Cách 1 (vét cạn) ta có thêm 3 cách: **(2)** dict + sort, **(3)** hai con trỏ, **(4)** mảng đếm theo chỉ số (đếm theo id).

### Cách 2 — Dict cộng dồn + sort

Giải thích từng đoạn:

1. `d = {}` là bảng tổng, key = id.
2. Hai vòng `for` đưa mọi cặp của `nums1` rồi `nums2` vào bảng bằng `d[key] = d.get(key, 0) + val`. Id chung tự được cộng lại.
3. `sorted(d.items())` sắp theo id, rồi đổi tuple thành list.

In [11]:
def merge_dict(nums1, nums2):
    d = {}
    for key, val in nums1:
        d[key] = d.get(key, 0) + val
    for key, val in nums2:
        d[key] = d.get(key, 0) + val
    return [[key, d[key]] for key in sorted(d)]

print(merge_dict([[1, 2], [2, 3], [4, 5]], [[1, 4], [3, 2], [4, 1]]))

[[1, 6], [2, 3], [3, 2], [4, 6]]


In [12]:
from typing import List

class SolutionDict:
    def mergeArrays(self, nums1: List[List[int]], nums2: List[List[int]]) -> List[List[int]]:
        d = {}
        for key, val in nums1:
            d[key] = d.get(key, 0) + val
        for key, val in nums2:
            d[key] = d.get(key, 0) + val
        return [[key, d[key]] for key in sorted(d)]

Độ phức tạp Cách 2: duyệt `O(n + m)`, sort `k` key `O(k log k)` → Time `O((n + m) log(n + m))`; Space `O(k)`.

### Cách 3 — Hai con trỏ (merge hai mảng đã sắp)

Giải thích từng đoạn:

1. `i`, `j` là vị trí đang xét ở `nums1`, `nums2`; `res` là kết quả.
2. Trong khi cả hai còn phần tử: so `id1 = nums1[i][0]`, `id2 = nums2[j][0]` và áp dụng ba nhánh `<`, `>`, `==`.
3. Khi vòng `while` kết thúc, **một** trong hai mảng đã hết; mảng kia có thể còn phần tử với id **lớn hơn** mọi id đã lấy → chép nốt bằng `res.extend(...)`. Dùng `[:]` copy từng cặp để không dùng chung list con với input.

In [13]:
def merge_two_pointers(nums1, nums2):
    i = j = 0
    res = []
    while i < len(nums1) and j < len(nums2):
        id1, v1 = nums1[i]
        id2, v2 = nums2[j]
        if id1 < id2:
            res.append([id1, v1])
            i += 1
        elif id1 > id2:
            res.append([id2, v2])
            j += 1
        else:
            res.append([id1, v1 + v2])
            i += 1
            j += 1
    while i < len(nums1):          # nums2 hết, chép nốt nums1
        res.append(nums1[i][:])
        i += 1
    while j < len(nums2):          # nums1 hết, chép nốt nums2
        res.append(nums2[j][:])
        j += 1
    return res

print(merge_two_pointers([[2, 4], [3, 6], [5, 5]], [[1, 3], [4, 3]]))

[[1, 3], [2, 4], [3, 6], [4, 3], [5, 5]]


In [14]:
from typing import List

class SolutionTwoPointers:
    def mergeArrays(self, nums1: List[List[int]], nums2: List[List[int]]) -> List[List[int]]:
        i = j = 0
        res = []
        while i < len(nums1) and j < len(nums2):
            id1, v1 = nums1[i]
            id2, v2 = nums2[j]
            if id1 < id2:
                res.append([id1, v1])
                i += 1
            elif id1 > id2:
                res.append([id2, v2])
                j += 1
            else:
                res.append([id1, v1 + v2])
                i += 1
                j += 1
        res.extend([k, v] for k, v in nums1[i:])
        res.extend([k, v] for k, v in nums2[j:])
        return res

Độ phức tạp Cách 3: mỗi con trỏ chỉ tiến, tổng bước `<= n + m` → Time `O(n + m)`; Space `O(1)` phụ (không kể mảng kết quả `O(n + m)`).

### Cách 4 — Mảng đếm theo chỉ số (counting array)

Do `id <= 1000`, dùng `total = [0] * 1001`. Cộng `val` vào `total[id]` cho mọi cặp của cả hai mảng, rồi quét `id = 1..1000`, lấy các id có `total[id] > 0` (vì `val >= 1`, id xuất hiện thì tổng chắc chắn `> 0`).

In [15]:
def merge_counting(nums1, nums2):
    total = [0] * 1001
    for key, val in nums1:
        total[key] += val
    for key, val in nums2:
        total[key] += val
    return [[key, total[key]] for key in range(1, 1001) if total[key] > 0]

print(merge_counting([[1, 2], [2, 3], [4, 5]], [[1, 4], [3, 2], [4, 1]]))

[[1, 6], [2, 3], [3, 2], [4, 6]]


In [16]:
from typing import List

class SolutionCounting:
    def mergeArrays(self, nums1: List[List[int]], nums2: List[List[int]]) -> List[List[int]]:
        total = [0] * 1001
        for key, val in nums1:
            total[key] += val
        for key, val in nums2:
            total[key] += val
        return [[key, total[key]] for key in range(1, 1001) if total[key] > 0]

Độ phức tạp Cách 4: Time `O(n + m + V)` với `V = 1000` (miền id); Space `O(V)`. Cách này dựa vào ràng buộc `id <= 1000`; nếu id lên tới `10^9` thì **không dùng được** (phải quay lại dict hoặc hai con trỏ).

## 8. TRACE từng bước

Trace Cách 3 (hai con trỏ) với `nums1 = [[1,2],[2,3],[4,5]]`, `nums2 = [[1,4],[3,2],[4,1]]`.

| bước | `i` | `j` | `nums1[i]` | `nums2[j]` | hành động | `res` sau bước |
|---|---|---|---|---|---|---|
| 1 | 0 | 0 | `[1,2]` | `[1,4]` | id bằng nhau → 2+4 | `[[1,6]]` |
| 2 | 1 | 1 | `[2,3]` | `[3,2]` | 2 < 3 → lấy bên 1 | `[[1,6],[2,3]]` |
| 3 | 2 | 1 | `[4,5]` | `[3,2]` | 4 > 3 → lấy bên 2 | `[[1,6],[2,3],[3,2]]` |
| 4 | 2 | 2 | `[4,5]` | `[4,1]` | bằng nhau → 5+1 | `[[1,6],[2,3],[3,2],[4,6]]` |

Sau bước 4: `i = 3`, `j = 3` → cả hai hết, không còn phần dư.

In [17]:
def trace_two_pointers(nums1, nums2):
    i = j = 0
    res = []
    step = 0
    while i < len(nums1) and j < len(nums2):
        step += 1
        i0, j0 = i, j                      # vị trí TRƯỚC khi di chuyển
        id1, v1 = nums1[i]
        id2, v2 = nums2[j]
        if id1 < id2:
            act = f"{id1} < {id2} -> lấy bên 1"
            res.append([id1, v1]); i += 1
        elif id1 > id2:
            act = f"{id1} > {id2} -> lấy bên 2"
            res.append([id2, v2]); j += 1
        else:
            act = f"{id1} == {id2} -> cộng {v1}+{v2}"
            res.append([id1, v1 + v2]); i += 1; j += 1
        print(f"bước {step}: i={i0} j={j0} | {act} | res={res}")
    print("phần dư nums1:", nums1[i:], "| phần dư nums2:", nums2[j:])
    res.extend(nums1[i:]); res.extend(nums2[j:])
    return res

print(trace_two_pointers([[1, 2], [2, 3], [4, 5]], [[1, 4], [3, 2], [4, 1]]))

bước 1: i=0 j=0 | 1 == 1 -> cộng 2+4 | res=[[1, 6]]
bước 2: i=1 j=1 | 2 < 3 -> lấy bên 1 | res=[[1, 6], [2, 3]]
bước 3: i=2 j=1 | 4 > 3 -> lấy bên 2 | res=[[1, 6], [2, 3], [3, 2]]
bước 4: i=2 j=2 | 4 == 4 -> cộng 5+1 | res=[[1, 6], [2, 3], [3, 2], [4, 6]]
phần dư nums1: [] | phần dư nums2: []
[[1, 6], [2, 3], [3, 2], [4, 6]]


In [18]:
# Trace ví dụ 2: không có id chung, nums2 hết trước -> thấy phần dư
print(trace_two_pointers([[2, 4], [3, 6], [5, 5]], [[1, 3], [4, 3]]))

bước 1: i=0 j=0 | 2 > 1 -> lấy bên 2 | res=[[1, 3]]
bước 2: i=0 j=1 | 2 < 4 -> lấy bên 1 | res=[[1, 3], [2, 4]]
bước 3: i=1 j=1 | 3 < 4 -> lấy bên 1 | res=[[1, 3], [2, 4], [3, 6]]
bước 4: i=2 j=1 | 5 > 4 -> lấy bên 2 | res=[[1, 3], [2, 4], [3, 6], [4, 3]]
phần dư nums1: [[5, 5]] | phần dư nums2: []
[[1, 3], [2, 4], [3, 6], [4, 3], [5, 5]]


## 9. Code nộp LeetCode

Ta nộp Cách 3 (hai con trỏ) vì nó tận dụng đúng điều kiện đề "đã sắp xếp" và nhanh `O(n + m)`. (Cách dict cũng được chấp nhận và ngắn hơn khi viết nhanh.)

In [19]:
from typing import List

class Solution:
    def mergeArrays(self, nums1: List[List[int]], nums2: List[List[int]]) -> List[List[int]]:
        i = j = 0
        res = []
        while i < len(nums1) and j < len(nums2):
            id1, v1 = nums1[i]
            id2, v2 = nums2[j]
            if id1 < id2:
                res.append([id1, v1])
                i += 1
            elif id1 > id2:
                res.append([id2, v2])
                j += 1
            else:
                res.append([id1, v1 + v2])
                i += 1
                j += 1
        while i < len(nums1):
            res.append([nums1[i][0], nums1[i][1]])
            i += 1
        while j < len(nums2):
            res.append([nums2[j][0], nums2[j][1]])
            j += 1
        return res

print(Solution().mergeArrays([[1, 2], [2, 3], [4, 5]], [[1, 4], [3, 2], [4, 1]]))
print(Solution().mergeArrays([[2, 4], [3, 6], [5, 5]], [[1, 3], [4, 3]]))

[[1, 6], [2, 3], [3, 2], [4, 6]]
[[1, 3], [2, 4], [3, 6], [4, 3], [5, 5]]


## 10. Vì sao thuật toán đúng

- **Cách hai con trỏ — invariant:** tại mọi thời điểm, `res` chứa đúng các id **nhỏ hơn** `min(nums1[i][0], nums2[j][0])` (hoặc mọi id đã xử lý), theo thứ tự tăng, mỗi id một lần với tổng đúng. Lý do: hai mảng tăng ngặt nên mọi phần tử chưa xử lý của `nums1` đều có id `>= nums1[i][0]`, của `nums2` đều `>= nums2[j][0]`.
- **Nhánh `<`:** `id1 < id2` ⇒ id1 nhỏ hơn mọi id còn lại của `nums2` ⇒ id1 **không** có trong `nums2`, giá trị tổng chính là `v1`; id1 cũng nhỏ nhất trong phần chưa xử lý nên chèn vào `res` giữ đúng thứ tự. Nhánh `>` đối xứng.
- **Nhánh `==`:** vì id **duy nhất** trong từng mảng, mỗi bên có đúng một phần tử mang id đó → tổng là `v1 + v2`; cả hai bên đều qua phần tử này nên tăng cả `i` và `j`.
- **Phần dư:** khi một mảng hết, các phần tử còn lại của mảng kia có id lớn hơn mọi id đã lấy và không trùng id nào ở mảng đã hết ⇒ chép nguyên vẹn là đúng.
- **Cách dict / mảng đếm:** mỗi cặp `[id, val]` được cộng đúng một lần vào ô của `id`, nên tổng của mỗi id bằng tổng ở hai mảng; sort / duyệt theo chỉ số cho đúng thứ tự.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums1)`, `m = len(nums2)`, `V = 1000` là miền id.

| Cách | Time | Space (phụ) |
|---|---|---|
| 1. Vét cạn (quét lại mỗi id) | `O((n + m)^2)` | `O(n + m)` |
| 2. Dict + sort | `O((n + m) log(n + m))` | `O(n + m)` |
| 3. Hai con trỏ | `O(n + m)` | `O(1)` (ngoài kết quả) |
| 4. Mảng đếm | `O(n + m + V)` | `O(V)` |

Với giới hạn `200` phần tử mọi cách đều tức thì; sự khác biệt chỉ quan trọng khi dữ liệu lớn.

## 12. Bộ test

Test 1: ví dụ đề, và các edge case: một phần tử mỗi mảng, id hoàn toàn khác nhau, id trùng hết, giá trị lớn nhất.

In [20]:
impls = [
    ("brute", merge_bruteforce),
    ("dict", merge_dict),
    ("two_pointers", merge_two_pointers),
    ("counting", merge_counting),
    ("solution_dict", SolutionDict().mergeArrays),
    ("solution_2p", SolutionTwoPointers().mergeArrays),
    ("solution_count", SolutionCounting().mergeArrays),
    ("submit", Solution().mergeArrays),
]
tests = [
    ([[1, 2], [2, 3], [4, 5]], [[1, 4], [3, 2], [4, 1]], [[1, 6], [2, 3], [3, 2], [4, 6]]),
    ([[2, 4], [3, 6], [5, 5]], [[1, 3], [4, 3]], [[1, 3], [2, 4], [3, 6], [4, 3], [5, 5]]),
    ([[1, 1]], [[1, 1]], [[1, 2]]),                          # cùng một id duy nhất
    ([[1, 1]], [[2, 1]], [[1, 1], [2, 1]]),                  # khác id, mỗi bên một phần tử
    ([[5, 7]], [[1, 3], [2, 4], [9, 9]], [[1, 3], [2, 4], [5, 7], [9, 9]]),   # nums1 xen giữa
    ([[1000, 1000]], [[1000, 1000]], [[1000, 2000]]),        # biên trên của id, val
    ([[1, 1], [2, 2], [3, 3]], [[1, 3], [2, 2], [3, 1]], [[1, 4], [2, 4], [3, 4]]),  # trùng toàn bộ id
]
for name, f in impls:
    for a, b, expected in tests:
        got = f([r[:] for r in a], [r[:] for r in b])
        assert got == expected, (name, a, b, got, expected)
    print(name, "PASS", len(tests), "test")

brute PASS 7 test
dict PASS 7 test
two_pointers PASS 7 test
counting PASS 7 test
solution_dict PASS 7 test
solution_2p PASS 7 test
solution_count PASS 7 test
submit PASS 7 test


Test 2: ngẫu nhiên có seed, **đúng constraints** (mỗi mảng 1..200 phần tử, id duy nhất tăng ngặt trong `1..1000`, `val` trong `1..1000`). Oracle độc lập: dùng `Counter` rồi sắp xếp.

In [21]:
import random
from collections import Counter

random.seed(2570)

def random_array(max_len):
    length = random.randint(1, max_len)
    ids = sorted(random.sample(range(1, 1001), length))
    return [[k, random.randint(1, 1000)] for k in ids]

def oracle(a, b):
    c = Counter()
    for k, v in a + b:
        c[k] += v
    return [[k, c[k]] for k in sorted(c)]

for round_no in range(300):
    max_len = random.choice([1, 2, 5, 20, 200])
    a, b = random_array(max_len), random_array(max_len)
    expected = oracle(a, b)
    for name, f in impls:
        got = f([r[:] for r in a], [r[:] for r in b])
        assert got == expected, (name, a, b)
print("300 test ngẫu nhiên: tất cả cách đều khớp oracle")

300 test ngẫu nhiên: tất cả cách đều khớp oracle


Test 3: tính chất — kết quả tăng ngặt theo id, tổng giá trị bảo toàn (`tổng kết quả = tổng nums1 + tổng nums2`), và hàm **không sửa** input.

In [22]:
random.seed(7)
for _ in range(300):
    a, b = random_array(30), random_array(30)
    a0 = [r[:] for r in a]; b0 = [r[:] for r in b]
    res = Solution().mergeArrays(a, b)
    ids = [k for k, _ in res]
    assert ids == sorted(set(ids)), "id phải tăng ngặt, không trùng"
    assert sum(v for _, v in res) == sum(v for _, v in a0) + sum(v for _, v in b0)
    assert a == a0 and b == b0, "không được sửa input"
    assert set(ids) == {k for k, _ in a0} | {k for k, _ in b0}
print("tính chất bảo toàn tổng, thứ tự, không sửa input: PASS")

tính chất bảo toàn tổng, thứ tự, không sửa input: PASS


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng `dict.update` — ghi đè thay vì cộng

In [23]:
def wrong_update(nums1, nums2):
    d = dict(nums1)           # dict([[1,2],[2,3]]) -> {1:2, 2:3}
    d.update(dict(nums2))     # id trùng bị GHI ĐÈ, không cộng
    return [[k, d[k]] for k in sorted(d)]

print(wrong_update([[1, 2], [2, 3], [4, 5]], [[1, 4], [3, 2], [4, 1]]))   # sai: id 1 ra 4, id 4 ra 1

[[1, 4], [2, 3], [3, 2], [4, 1]]


Vì sao sai: `update` chỉ thay giá trị cũ bằng giá trị mới (id 1 phải là `2 + 4 = 6` nhưng ra `4`). Phải cộng: `d[key] = d.get(key, 0) + val`.

In [24]:
def right_update(nums1, nums2):
    d = {}
    for key, val in nums1 + nums2:
        d[key] = d.get(key, 0) + val
    return [[k, d[k]] for k in sorted(d)]

assert right_update([[1, 2], [2, 3], [4, 5]], [[1, 4], [3, 2], [4, 1]]) == [[1, 6], [2, 3], [3, 2], [4, 6]]
print("đã sửa: PASS")

đã sửa: PASS


### Lỗi 2: quên chép phần dư sau vòng `while` của hai con trỏ

In [25]:
def wrong_no_leftover(nums1, nums2):
    i = j = 0
    res = []
    while i < len(nums1) and j < len(nums2):
        if nums1[i][0] < nums2[j][0]:
            res.append(nums1[i][:]); i += 1
        elif nums1[i][0] > nums2[j][0]:
            res.append(nums2[j][:]); j += 1
        else:
            res.append([nums1[i][0], nums1[i][1] + nums2[j][1]]); i += 1; j += 1
    return res                # THIẾU phần dư

print(wrong_no_leftover([[2, 4], [3, 6], [5, 5]], [[1, 3], [4, 3]]))   # mất [5, 5]

[[1, 3], [2, 4], [3, 6], [4, 3]]


Vì sao sai: khi `nums2` hết ở `j = 2`, `nums1` vẫn còn `[5, 5]` mà vòng `while` đã dừng. Phải chép nốt: `res.extend(nums1[i:])` và `res.extend(nums2[j:])`.

### Lỗi 3: quên sắp xếp kết quả của dict

In [26]:
def wrong_no_sort(nums1, nums2):
    d = {}
    for key, val in nums1 + nums2:
        d[key] = d.get(key, 0) + val
    return [[k, v] for k, v in d.items()]     # thứ tự chèn, không phải tăng dần

print(wrong_no_sort([[2, 4], [3, 6], [5, 5]], [[1, 3], [4, 3]]))   # id 1 (từ nums2) ra sau cùng 2, 3, 5

[[2, 4], [3, 6], [5, 5], [1, 3], [4, 3]]


Vì sao sai: dict nhớ **thứ tự chèn** chứ không phải thứ tự theo key. Id `1` chỉ có ở `nums2` nên bị chèn sau `2, 3, 5`. Đề yêu cầu tăng dần theo id → luôn `sorted(d)`.

### Lỗi 4: trả về list các tuple

In [27]:
res_tuple = sorted({1: 6, 2: 3}.items())
print(res_tuple, "==", [[1, 6], [2, 3]], "?", res_tuple == [[1, 6], [2, 3]])   # False: tuple != list
print([list(p) for p in res_tuple] == [[1, 6], [2, 3]])                          # True

[(1, 6), (2, 3)] == [[1, 6], [2, 3]] ? False
True


Vì sao sai: `(1, 6) == [1, 6]` là `False` trong Python. Đề trả về mảng 2 chiều, nên đổi tuple thành list: `[[k, v] for k, v in sorted(d.items())]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Gộp hai danh sách theo **khoá**, cộng giá trị cùng khoá" → **dict cộng dồn** (`d.get(k, 0) + v`) hoặc `Counter`.
- Thấy cụm "**sắp xếp tăng dần theo id**" ở cả đầu vào lẫn đầu ra → nghĩ ngay **hai con trỏ / merge**; đây là bước con của merge sort.
- Thấy khoá có miền **nhỏ** (`<= 1000`) → **mảng đếm theo chỉ số** để khỏi sort.
- Muốn code ngắn khi thi: một vòng `for key, val in nums1 + nums2` cộng vào `Counter` hoặc dict rồi `sorted` là đủ; `Counter(dict(nums1)) + Counter(dict(nums2))` cũng chạy được vì mọi `val >= 1` (phép `+` của `Counter` bỏ giá trị `<= 0`).
- Đừng đặt tên biến `id` (che mất hàm `id` có sẵn) — dùng `key`, `uid`, `k`.

## 15. Biến thể của bài

1. Gộp **k** mảng (không chỉ hai): dùng dict cộng dồn, hoặc gộp bằng heap (`heapq.merge`) khi mỗi mảng đã sắp.
2. Thay "cộng" bằng **lấy max / lấy min / lấy hiệu** giữa hai giá trị cùng id.
3. Chỉ giữ các id xuất hiện ở **cả hai** mảng (giao thay vì hợp).
4. Id **không** bị giới hạn `1000` mà lên tới `10^9` (cách mảng đếm không dùng được).
5. Mảng đầu vào **chưa sắp xếp** và/hoặc id có thể lặp ngay trong một mảng (phải dùng dict).
6. Phiên bản Linked List: "Merge Two Sorted Lists" (21) — cùng kỹ thuật hai con trỏ.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | với mỗi id, quét lại cả hai mảng | `O((n+m)^2)` | `O(n+m)` | chỉ để hiểu đề / làm oracle |
| 2. Dict + sort | cộng dồn theo key rồi sort key | `O((n+m) log(n+m))` | `O(n+m)` | mảng **chưa** sắp, viết nhanh, an toàn |
| 3. Hai con trỏ | merge hai mảng đã sắp | `O(n+m)` | `O(1)` phụ | khi đề cho sẵn thứ tự (bài này) |
| 4. Mảng đếm | `total[id] += val`, quét `1..V` | `O(n+m+V)` | `O(V)` | khi miền id nhỏ và cố định |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().mergeArrays([[1, 5]], [[1, 5], [2, 1]])` bằng gì?

**Bài 2 (tính tay):** gộp `nums1 = [[1,1],[3,3],[5,5]]` và `nums2 = [[2,2],[3,3],[6,6]]`; viết ra kết quả và lập bảng `i, j, hành động` như mục 8.

**Bài 3 (điền TODO):** hoàn thành `common_ids(nums1, nums2)` trả về **danh sách các id có ở cả hai mảng** (tăng dần), dùng hai con trỏ.

**Bài 4 (tự viết lại):** viết `merge_max(nums1, nums2)`: gộp theo id nhưng giá trị là **max** của hai giá trị (id thiếu ở một bên → lấy giá trị bên có).

**Bài 5 (mini challenge):** viết `merge_k(arrays)` nhận danh sách **k** mảng dạng `[id, val]` (mỗi mảng id duy nhất) và trả về mảng gộp cộng dồn, tăng theo id.

In [28]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().mergeArrays([[1, 5]], [[1, 5], [2, 1]]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def common_ids(nums1, nums2):
    i = j = 0
    result = []
    # TODO: while i < len(nums1) and j < len(nums2):
    #   - nếu id nums1[i][0] < id nums2[j][0] -> i += 1
    #   - nếu lớn hơn -> j += 1
    #   - nếu bằng nhau -> thêm id vào result và tăng cả i, j
    return result

[[1, 10], [2, 1]]


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 3: cùng khung với hai con trỏ ở mục 8, nhưng nhánh "bằng nhau" mới ghi kết quả; hai nhánh còn lại chỉ nhích một con trỏ, không ghi gì. Bài 4: hai mảng id trùng thì chọn `max`.

**Mức 2 (vừa):** Bài 4: có thể dùng dict `d[key] = max(d.get(key, 0), val)` vì mọi `val >= 1`. Bài 5: duyệt mọi cặp của mọi mảng vào một dict cộng dồn rồi `sorted`.

**Mức 3 (gần đáp án):** Bài 2: id 1 → 1 (chỉ nums1), 2 → 2 (chỉ nums2), 3 → 3+3=6, 5 → 5, 6 → 6. Kết quả `[[1,1],[2,2],[3,6],[5,5],[6,6]]`. Bài 1: id 1 → 5+5=10, id 2 → 1 ⇒ `[[1,10],[2,1]]`.

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1
assert Solution().mergeArrays([[1, 5]], [[1, 5], [2, 1]]) == [[1, 10], [2, 1]]

# Bài 2
assert Solution().mergeArrays([[1, 1], [3, 3], [5, 5]], [[2, 2], [3, 3], [6, 6]]) == [[1, 1], [2, 2], [3, 6], [5, 5], [6, 6]]

# Bài 3
def common_ids_answer(nums1, nums2):
    i = j = 0
    result = []
    while i < len(nums1) and j < len(nums2):
        if nums1[i][0] < nums2[j][0]:
            i += 1
        elif nums1[i][0] > nums2[j][0]:
            j += 1
        else:
            result.append(nums1[i][0])
            i += 1
            j += 1
    return result

assert common_ids_answer([[1, 1], [3, 3], [5, 5]], [[2, 2], [3, 3], [5, 6]]) == [3, 5]
assert common_ids_answer([[1, 1]], [[2, 2]]) == []

# Bài 4
def merge_max(nums1, nums2):
    d = {}
    for key, val in nums1 + nums2:
        d[key] = max(d.get(key, 0), val)
    return [[k, d[k]] for k in sorted(d)]

assert merge_max([[1, 4], [2, 9]], [[1, 7], [3, 1]]) == [[1, 7], [2, 9], [3, 1]]

# Bài 5
def merge_k(arrays):
    d = {}
    for arr in arrays:
        for key, val in arr:
            d[key] = d.get(key, 0) + val
    return [[k, d[k]] for k in sorted(d)]

assert merge_k([[[1, 1], [2, 2]], [[2, 3]], [[1, 5], [4, 4]]]) == [[1, 6], [2, 5], [4, 4]]
assert merge_k([[[7, 1]]]) == [[7, 1]]
print("tất cả đáp án bài tập: PASS")

tất cả đáp án bài tập: PASS


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template A — hai mảng đã sắp, merge bằng hai con trỏ:**

```python
i = j = 0
res = []
while i < len(nums1) and j < len(nums2):
    id1, v1 = nums1[i]
    id2, v2 = nums2[j]
    if id1 < id2:   res.append([id1, v1]); i += 1
    elif id1 > id2: res.append([id2, v2]); j += 1
    else:           res.append([id1, v1 + v2]); i += 1; j += 1
res.extend(nums1[i:]); res.extend(nums2[j:])
return res
```

**Template B — dict cộng dồn:**

```python
d = {}
for key, val in nums1 + nums2:
    d[key] = d.get(key, 0) + val
return [[k, d[k]] for k in sorted(d)]
```

**Checklist trước khi submit:**

- [ ] Id trùng thì **cộng**, không ghi đè?
- [ ] Kết quả **tăng dần theo id** (đã sort hoặc đi bằng hai con trỏ)?
- [ ] Sau vòng `while` đã **chép phần dư** của cả hai mảng?
- [ ] Trả về **list các list** `[id, tổng]`, không phải tuple?
- [ ] Không đặt tên biến `id`; không sửa input?
- [ ] Đã thử: một phần tử mỗi mảng, hai mảng không id chung, hai mảng id trùng hoàn toàn, id/val = 1000?
- [ ] Chữ ký đúng `mergeArrays(self, nums1: List[List[int]], nums2: List[List[int]]) -> List[List[int]]`?